# Domain-specific LLM Agents

## Developing and Managing Domain-specific LLM Agents

*D. Brian Letort, PhD — Head of Data Office & Platform AI, Author, & Adjunct Professor*

---

## Course Overview

| # | Module | Topic |
|---|--------|-------|
| 01 | Introduction to LLM Agents | What are LLMs, general-purpose vs domain-specific, comparison demo |
| 02 | Curating and Preparing Domain-specific Datasets | Data collection, preparation strategies, preprocessing pipeline demo |
| 03 | Fine-tuning Techniques for Domain Applications | Fine-tuning methods (SFT, LoRA, PEFT), domain-specific LLM demo |
| 04 | Ensuring Compliance, Privacy, and Continuous Improvement | Privacy, governance, continuous learning, feedback loop demo |

**Version Check — libraries used in this notebook:**
- Python 3.14.6
- torch 2.14.0
- transformers 4.57.6
- datasets 5.0.1
- accelerate 1.15.0
- peft 0.20.0
- trl 1.13.0
- openai 3.11.0
- python-dotenv 1.2.3
- presidio-analyzer 2.2.364
- presidio-anonymizer 2.2.364
- PyCharm 2025.1

---

# Module 01 — Introduction to LLM Agents

This module introduces Large Language Models, contrasts general-purpose and domain-specific LLMs, and shows hands-on why specialisation matters.

## 1.1 Introduction to Large Language Models (LLMs)

Large Language Models are deep-learning models trained on massive text corpora to understand and generate human language.

There are two broad categories:

| Type | Description | Examples |
|------|-------------|----------|
| **General-purpose LLMs** | Trained on diverse internet-scale data; can answer almost any question | GPT-4, Claude, Gemini |
| **Domain-specific LLMs** | Fine-tuned on curated data from a narrow field; optimised for accuracy in that domain | Med-PaLM (medicine), BloombergGPT (finance), custom models |

### Why Domain-specific LLMs?

General-purpose models are **flexible but shallow** in niche areas. A domain-specific model:
- Understands field-specific terminology out of the box
- Adheres to domain conventions (clinical, legal, financial)
- Reduces hallucinations on specialised questions
- Can be kept up-to-date with domain developments

## 1.2 Key Differences Between General-purpose and Domain-specific LLMs

| Dimension | General-purpose | Domain-specific |
|-----------|----------------|-----------------|
| **Applications** | Broad conversational AI, content generation | Specialized assistance (e.g., troubleshooting data center issues) |
| **Strengths** | Flexibility, general knowledge | Higher accuracy, context awareness, compliance adherence |
| **Limitations** | Prone to hallucinations in niche areas | Requires high-quality domain data, regular updates |

## 1.3 Demo: Comparing General-purpose vs Domain-specific LLM

This demo shows the difference in responses between a general-purpose LLM and a domain-specific LLM fine-tuned on **data center operations**.

We simulate a domain-specific model by using a heavily-scoped system prompt — a practical stand-in when you do not have a fine-tuned checkpoint available locally.

> **Prerequisites:** Set `OPENAI_API_KEY` in a `.env` file before running.

In [ ]:
%pip install -q -U openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY not set — add it to your .env file"

In [ ]:
from openai import OpenAI

client = OpenAI()

def ask(system_prompt: str, user_message: str, label: str) -> str:
    """Send a message and print the response with a label."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
        temperature=0,
    )
    answer = response.choices[0].message.content
    print(f"=== {label} ===")
    print(answer)
    print()
    return answer

In [ ]:
GENERAL_SYSTEM = "You are a helpful general-purpose AI assistant."

DOMAIN_SYSTEM = """
You are a domain-specific AI assistant specialised in data center operations.
You have deep expertise in:
  - Server hardware diagnostics and failure patterns
  - Network topology, VLAN configuration, and BGP routing issues
  - Power distribution (PDU, UPS) and thermal management (CRAC units, hot/cold aisles)
  - Incident response procedures and SLA escalation workflows
  - Compliance requirements: ISO 27001, SOC 2, GDPR for data center environments

Always respond with actionable, technically precise guidance.
When diagnosing issues, follow a structured root-cause-analysis approach.
"""

question = "One of our server racks is running at 40°C inlet temperature. What should I check first?"

ask(GENERAL_SYSTEM, question, "General-purpose LLM")
ask(DOMAIN_SYSTEM,  question, "Domain-specific LLM (Data Center)")

### What to observe

The general-purpose model gives a reasonable answer, but it is broad and non-specific.

The domain-specific model:
- Uses correct terminology (CRAC unit, hot/cold aisle, inlet temperature thresholds)
- Follows a structured diagnostic sequence matching real-world runbooks
- References compliance implications if left unresolved

This difference becomes even more pronounced when you actually fine-tune a model on curated domain corpora rather than just scoping the system prompt.

---

# Module 02 — Curating and Preparing Domain-specific Datasets

The quality of a domain-specific model is determined almost entirely by its training data. Garbage in, garbage out — this module covers how to collect, clean, and format data for LLM fine-tuning.

## 2.1 Data Collection and Preparation Strategies

### Data Collection Methods

- Utilise open datasets and public repositories (Hugging Face Hub, academic datasets)
- Ensure **data diversity** to cover various scenarios within the domain
- Collect from internal sources: runbooks, ticket histories, documentation, expert annotations

### Data Preparation Steps

| Step | What to do |
|------|------------|
| **Cleaning** | Remove inconsistencies, duplicates, and irrelevant information |
| **Formatting** | Ensure compatibility with LLM training requirements (e.g., instruction-following format) |
| **Annotation** | Highlight domain-specific terminology and context; add instruction–response pairs |

### Importance of High-quality Data

- Enhances model accuracy and relevance
- Reduces biases and errors in model predictions
- A small, high-quality domain dataset often outperforms a large, noisy one

## 2.2 Demo: Implement a Data Preprocessing Pipeline

We build a preprocessing pipeline that takes raw domain text, cleans it, and formats it into the **instruction–response pairs** expected by supervised fine-tuning.

The output format follows the standard used by Hugging Face `transformers` trainers:
```json
{"instruction": "...", "response": "..."}
```

In [ ]:
%pip install -q -U datasets transformers

In [ ]:
# Simulated raw domain data — in practice this would come from runbooks, tickets, etc.
RAW_DOMAIN_DATA = [
    {
        "question": "  How do I reset a PDU outlet remotely?  ",
        "answer": "Use the iLO/IPMI interface: navigate to Power Management > PDU Outlets, \nselect the target outlet, and click Cycle.   "
    },
    {
        "question": "server won't POST after firmware update",
        "answer": "1. Clear CMOS. 2. Re-flash firmware via USB recovery image. 3. Check iLO event log for hardware fault codes."
    },
    {
        "question": "",  # empty — should be filtered out
        "answer": "N/A"
    },
    {
        "question": "What VLAN should storage traffic use?",
        "answer": "Storage traffic (iSCSI/NFS) should be isolated on a dedicated VLAN (e.g., VLAN 200) with jumbo frames (MTU 9000) enabled end-to-end."
    },
]

print(f"Raw records: {len(RAW_DOMAIN_DATA)}")

In [ ]:
import re
import json

def clean_text(text: str) -> str:
    """Strip leading/trailing whitespace and collapse internal whitespace."""
    text = text.strip()
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n+', '\n', text)
    return text

def is_valid(record: dict) -> bool:
    """Filter out records with empty or placeholder content."""
    return bool(record["question"].strip()) and record["answer"].strip() not in ("", "N/A")

def to_instruction_pair(record: dict) -> dict:
    """Convert a raw Q&A record to an instruction–response pair."""
    return {
        "instruction": clean_text(record["question"]),
        "response":    clean_text(record["answer"]),
    }

# Run the pipeline
processed = [
    to_instruction_pair(r)
    for r in RAW_DOMAIN_DATA
    if is_valid(r)
]

print(f"Records after cleaning: {len(processed)}\n")
for p in processed:
    print(json.dumps(p, indent=2))
    print()

In [ ]:
from datasets import Dataset

# Wrap into a Hugging Face Dataset — the standard input for the Trainer API
ds = Dataset.from_list(processed)
print(ds)
print()
print(ds[0])

### Instruction format for fine-tuning

Many models expect a specific prompt template during fine-tuning so the model learns the correct input/output boundary.
A common format (used by Alpaca, Mistral-Instruct, and others):

```
### Instruction:
{instruction}

### Response:
{response}
```

Always check the base model's documentation for its expected template before formatting your dataset.

In [ ]:
def apply_prompt_template(example: dict) -> dict:
    """Wrap each pair in the Alpaca-style instruction template."""
    text = (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Response:\n{example['response']}"
    )
    return {"text": text}

ds_formatted = ds.map(apply_prompt_template)
print(ds_formatted[0]["text"])

---

# Module 03 — Fine-tuning Techniques for Domain Applications

Fine-tuning adapts a pre-trained base model to a specific domain by continuing training on your curated dataset. This module covers the three main approaches and shows a complete fine-tuning run using **Mistral-7B** and **LoRA**.

## 3.1 Fine-tuning Domain-specific LLMs

Fine-tuning starts from a **base LLM** (pre-trained on broad data) and continues training on domain-specific data to produce a **fine-tuned LLM** that excels in the target domain.

```
Domain-specific Data
         │
         ▼
   Base LLM  ──────────►  Fine-tuned LLM
```

### Steps

1. **Dataset Selection** — curated, cleaned instruction–response pairs (Module 02)
2. **Model Selection** — choose a base model appropriate for the domain and compute budget
3. **Fine-tuning Process** — train with appropriate technique (see below)
4. **Evaluation** — measure on held-out domain benchmarks; compare against the base model

## 3.2 Fine-tuning Methods

| Method | Description | When to use |
|--------|-------------|-------------|
| **Supervised Fine-tuning (SFT)** | Update all model weights on labeled instruction–response pairs | When you have significant compute and a large, high-quality dataset |
| **LoRA** (Low-rank Adaptation) | Freeze base weights; train small low-rank adapter matrices injected into attention layers | Most practical choice: ~10–100× fewer trainable parameters, runs on consumer GPUs |
| **PEFT** (Parameter-efficient Fine-tuning) | Umbrella term covering LoRA, prefix-tuning, adapter layers, etc. | When GPU memory is constrained; LoRA is the most popular PEFT method |

### Why LoRA?

A 7B-parameter model has ~28 GB of weights. Full fine-tuning requires gradient storage on top of that — impractical on most hardware. LoRA injects trainable matrices of rank `r` (e.g., `r=8`) into each attention projection:

```
W_adapted = W_frozen + B × A
            (7B params) (r × d  +  d × r  ≈ millions, not billions)
```

Only `A` and `B` are trained. After training they can be merged back into `W` for zero-overhead inference.

## 3.3 Demo: Building a Domain-specific LLM

**Fine-tuning a Fitness and Nutrition Model using Mistral-7B**

We use:
- `mistralai/Mistral-7B-Instruct-v0.1` as the base model
- `peft` + `trl` for LoRA-based SFT
- A small fitness & nutrition Q&A dataset (simulated here; swap for a real Hugging Face dataset)

> **Hardware note:** Running this demo requires a GPU with at least 16 GB VRAM (e.g., A100, RTX 3090).  
> On CPU-only machines, set `device_map="cpu"` and reduce `max_steps` — it will be slow but will run.

In [ ]:
%pip install -q -U transformers peft trl datasets accelerate bitsandbytes

In [ ]:
from datasets import Dataset

# Simulated fitness & nutrition Q&A pairs
fitness_data = [
    {"instruction": "How many calories does a 30-minute run burn?",
     "response": "A 30-minute run burns roughly 300–400 kcal for a 70 kg person at a moderate pace (8–10 km/h). Use the formula: METs × weight(kg) × duration(hours). Running at 9 km/h has a MET of ~9.8."},
    {"instruction": "What is the optimal protein intake for muscle gain?",
     "response": "Research supports 1.6–2.2 g of protein per kg of body weight per day for maximising muscle protein synthesis. Distribute intake evenly across 3–5 meals (30–40 g per serving) to optimise leucine-triggered anabolic signalling."},
    {"instruction": "Should I do cardio before or after weight training?",
     "response": "If your goal is strength/hypertrophy, do weights first — glycogen stores and neuromuscular performance are highest before cardio fatigue sets in. If endurance is the primary goal, reverse the order. For general fitness, either order works; consistency matters more than sequence."},
    {"instruction": "What are the signs of overtraining syndrome?",
     "response": "Key signs: persistent fatigue despite rest, declining performance over weeks, elevated resting heart rate (>5 bpm above baseline), mood disturbances, increased injury frequency, and disrupted sleep. Treatment: structured deload (50% volume reduction for 1–2 weeks) and prioritise sleep and nutrition."},
]

def format_mistral_instruct(example):
    text = f"[INST] {example['instruction']} [/INST] {example['response']}"
    return {"text": text}

ds_fitness = Dataset.from_list(fitness_data).map(format_mistral_instruct)
print(f"Training examples: {len(ds_fitness)}")
print(ds_fitness[0]["text"])

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.1"

# 4-bit quantisation — reduces GPU memory from ~14 GB to ~4 GB
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # required for batch training

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Model loaded: {MODEL_ID}")
print(f"Parameters: {model.num_parameters() / 1e9:.1f}B")

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for 4-bit training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,                    # rank of adapter matrices — higher = more capacity, more params
    lora_alpha=16,          # scaling factor (alpha/r = 2 is a common default)
    target_modules=["q_proj", "v_proj"],  # inject into query and value projections
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./mistral-fitness-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    max_seq_length=512,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds_fitness,
    args=training_args,
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model()

In [ ]:
# Load the fine-tuned adapter for inference
from peft import PeftModel

ft_model = PeftModel.from_pretrained(model, "./mistral-fitness-lora")
ft_model.eval()

prompt = "[INST] How should I structure a weekly workout plan for fat loss? [/INST]"
inputs = tokenizer(prompt, return_tensors="pt").to(ft_model.device)

with torch.no_grad():
    outputs = ft_model.generate(**inputs, max_new_tokens=200, temperature=0.7)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

---

# Module 04 — Ensuring Compliance, Privacy, and Continuous Improvement

Deploying a domain-specific LLM in a regulated environment requires more than a good model. This module covers the governance, privacy, and continuous learning practices that keep a domain LLM safe, compliant, and improving over time.

## 4.1 Ensuring Compliance, Privacy, and Continuous Learning

### Data Privacy & Compliance

| Regulatory framework | Key requirement | Mitigation |
|----------------------|----------------|------------|
| **GDPR** | Right to erasure, data minimisation | Anonymise PII before ingestion; maintain data lineage |
| **HIPAA** | Protected Health Information (PHI) | De-identification pipeline; access-controlled model serving |
| **CCPA** | Consumer data rights | Consent tracking; opt-out mechanisms for training data |

**Mitigations:** data anonymisation, differential privacy, role-based access control (RBAC).

### Governance & Auditing

- Adopt AI model governance frameworks such as the **NIST AI Risk Management Framework (AI RMF)**
- Maintain: logging of all inference calls, model version control, access tracking
- Conduct periodic bias audits on domain-specific outputs

### Continuous Learning & Feedback Loops

- Implement **human-in-the-loop** for model adaptation on ambiguous or high-stakes outputs
- Leverage user feedback for iterative fine-tuning (small batches of corrected examples)
- Auto-curate domain-specific datasets from real interactions (with anonymisation)

## 4.2 Demo: Implementing a Privacy-preserving Feedback Loop

**Using Anonymized Data in Continuous Learning to update the Domain-specific Model**

This demo shows a minimal but complete pipeline:
1. User interacts with the model and provides feedback
2. PII is detected and anonymised before the feedback is stored
3. Anonymised feedback pairs are accumulated into a new fine-tuning batch
4. The batch is used to update the LoRA adapter (continuous learning)


In [ ]:
%pip install -q -U presidio-analyzer presidio-anonymizer spacy
# Download the English NLP model for Presidio
import subprocess
subprocess.run(["python", "-m", "spacy", "download", "en_core_web_lg"], check=True)

In [ ]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

analyzer   = AnalyzerEngine()
anonymizer = AnonymizerEngine()

def anonymise(text: str) -> str:
    """Detect and replace PII entities with placeholders."""
    results = analyzer.analyze(text=text, language="en")
    return anonymizer.anonymize(text=text, analyzer_results=results).text

# Test the anonymiser
sample = "My name is John Smith and my email is john.smith@example.com. I weigh 85 kg."
print("Original:",    sample)
print("Anonymised:", anonymise(sample))

In [ ]:
import json
from pathlib import Path

FEEDBACK_FILE = Path("feedback_buffer.jsonl")

def store_feedback(user_input: str, model_response: str, user_correction: str):
    """
    Anonymise the user correction and append it to the feedback buffer.
    In production this would write to a secure, access-controlled store.
    """
    record = {
        "instruction": anonymise(user_input),
        "response":    anonymise(user_correction),
    }
    with FEEDBACK_FILE.open("a") as f:
        f.write(json.dumps(record) + "\n")
    print(f"Stored (anonymised): {record}")

# Simulate a few feedback interactions
store_feedback(
    user_input="What protein powder should I use?",
    model_response="Whey concentrate is a good all-rounder.",
    user_correction="Whey isolate is better for lactose-sensitive individuals; casein suits slow-release overnight protein needs.",
)

store_feedback(
    user_input="My trainer Alice Johnson recommends 5×5. Is that good for beginners?",
    model_response="5×5 is effective for intermediate lifters.",
    user_correction="5×5 (e.g., StrongLifts) is well-suited for beginners as well as intermediate lifters due to its focus on progressive overload and compound movements.",
)

In [ ]:
from datasets import Dataset

def load_feedback_buffer(path: Path) -> Dataset:
    """Load accumulated feedback pairs as a Hugging Face Dataset."""
    records = [json.loads(line) for line in path.read_text().strip().splitlines()]
    return Dataset.from_list(records).map(format_mistral_instruct)

if FEEDBACK_FILE.exists() and FEEDBACK_FILE.stat().st_size > 0:
    ds_feedback = load_feedback_buffer(FEEDBACK_FILE)
    print(f"Feedback records available for continuous update: {len(ds_feedback)}")
    for row in ds_feedback:
        print("-", row["text"][:120], "...")
else:
    print("No feedback yet.")

### Continuous update strategy

Once the feedback buffer reaches a threshold (e.g., 50–100 new pairs), trigger a short LoRA update run:

```python
# Re-use the SFTTrainer from Module 03, passing ds_feedback as train_dataset
trainer_continuous = SFTTrainer(
    model=ft_model,
    train_dataset=ds_feedback,
    args=SFTConfig(
        output_dir="./mistral-fitness-lora-v2",
        num_train_epochs=1,      # short update, not full retraining
        max_steps=20,
        ...
    ),
)
trainer_continuous.train()
```

**Key safeguards:**
- Always run the anonymisation step before storing feedback
- Validate new examples with a human reviewer before including them in training
- Version the adapter checkpoint so you can roll back if quality regresses
- Monitor model outputs with automated evaluation on a held-out domain benchmark after each update

---

# Course Summary

| Module | Key concept | Tools used |
|--------|-------------|------------|
| **01 — Introduction to LLM Agents** | General-purpose vs domain-specific LLMs; why specialisation reduces hallucinations | OpenAI API, python-dotenv |
| **02 — Curating and Preparing Datasets** | Clean → format → annotate pipeline; instruction–response pairs | Hugging Face `datasets`, custom preprocessing |
| **03 — Fine-tuning Techniques** | SFT, LoRA, PEFT; 4-bit quantisation; LoRA adapter training | `transformers`, `peft`, `trl`, `bitsandbytes` |
| **04 — Compliance, Privacy, Continuous Learning** | GDPR/HIPAA/CCPA mitigations; feedback loop with PII anonymisation | Presidio, continuous LoRA updates |

These four modules are the end-to-end lifecycle of a domain-specific LLM: understand the need → gather and clean data → train the model → deploy it safely and keep it improving.

---

## Further Reading

- 📄 [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685) — Hu et al., 2021
- 📄 [PEFT: Parameter-Efficient Fine-Tuning of Billion-Scale Models](https://huggingface.co/blog/peft)
- 📖 [Hugging Face TRL — SFTTrainer documentation](https://huggingface.co/docs/trl/sft_trainer)
- 📖 [Microsoft Presidio — PII anonymisation](https://microsoft.github.io/presidio/)
- 📖 [NIST AI Risk Management Framework](https://www.nist.gov/system/files/documents/2023/01/26/AI%20RMF%201.0.pdf)